## Labeling Target by Summary Statistics

1. Load preprocessed dataset
2. Create Labeled and Unlabeled Data
3. Generate Summary Statistics from Labeled Data 
4. Find closest labeled observation using Euclidean distance
5. Semi-supervised Learning / Pseudo-labeling
6. Lọc nhãn giả → Gộp vào tập train → Huấn luyện lại (Self-training)
7. So sánh các phương pháp
---

##### 1. Load Preprocessed Dataset

In [1]:
import pandas as pd

df = pd.read_csv("../data/loan_preprocessed.csv")

print(df.shape)
df.head()

(100000, 13)


,all_util,bc_open_to_buy,percent_bc_gt_75,bc_util,annual_inc,loan_amnt,dti,term,grade,sub_grade,verification_status,home_ownership,int_rate
0,73.000000,8.992557,50.0,57.9,11.097425,16000,18.62,36 months,B,B3,Verified,MORTGAGE,10.42
1,83.000000,6.278521,100.0,83.4,10.933125,6000,11.74,36 months,C,C4,Not Verified,OWN,15.05
2,52.000000,10.053071,22.2,52.0,11.156265,15000,12.63,36 months,C,C1,Source Verified,OWN,12.74
3,57.015308,10.389365,33.3,44.5,11.512935,27000,10.63,36 months,A,A1,Not Verified,MORTGAGE,5.32
4,57.015308,7.655391,0.0,8.2,10.714440,1600,15.23,36 months,D,D1,Verified,MORTGAGE,15.61


##### 2. Create Labeled and Unlabeled Data
Để mô phỏng bài toán gán nhãn cho dữ liệu hồi quy, một phần giá trị của biến mục tiêu `int_rate` được che đi và xem như dữ liệu chưa có nhãn (unlabeled data).

Trong ví dụ này, 80% dữ liệu được chọn ngẫu nhiên để loại bỏ nhãn. Phần dữ liệu còn lại được sử dụng làm tập dữ liệu đã gán nhãn (labeled data).

In [2]:
import numpy as np

np.random.seed(42)

df_labeling = df.copy()

n_unlabeled = int(len(df_labeling) * 0.8)

random_index = np.random.choice(
    df_labeling.index,
    size=n_unlabeled,
    replace=False
)

df_labeling.loc[random_index, "int_rate"] = np.nan

labeled_df = df_labeling[
    df_labeling["int_rate"].notna()
]

unlabeled_df = df_labeling[
    df_labeling["int_rate"].isna()
]

print("Labeled:", len(labeled_df))
print("Unlabeled:", len(unlabeled_df))

Labeled: 20000
Unlabeled: 80000


##### 3. Generate Summary Statistics from Labeled Data
Trong sách dùng cách tính summary statistics (thống kê mô tả) cho toàn bộ tập`int_rate`
Từ đó, mỗi giá trị


`int_rate`


↓


Có một bộ thống kê đặc trưng đi kèm (feature profile)
Ví dụ: 


int_rate = 10.99 </br>
```text
loan_amnt
├─ mean
├─ std
├─ min
├─ max

annual_inc
├─ mean
├─ std
├─ min
├─ max

```
int_rate = 11.99 </br>
```text
loan_amnt
├─ mean
├─ std
├─ min
├─ max

annual_inc
├─ mean
├─ std
├─ min
├─ max

```

In [3]:
summary_stats = labeled_df.groupby('int_rate').describe()
summary_stats.head()

all_util                                                            \
            count       mean           std        min        25%        50%   
int_rate                                                                      
5.31         81.0  41.148148  2.134778e+01   0.000000  27.000000  41.000000   
5.32        412.0  41.645508  1.807700e+01   0.000000  28.000000  41.500000   
5.42          7.0  57.015308  0.000000e+00  57.015308  57.015308  57.015308   
5.79          4.0  57.015308  0.000000e+00  57.015308  57.015308  57.015308   
5.93          9.0  57.015308  7.536444e-15  57.015308  57.015308  57.015308   

                               bc_open_to_buy             ... loan_amnt  \
                75%        max          count       mean  ...       75%   
int_rate                                                  ...             
5.31      57.000000  90.000000           81.0   9.958363  ...  19000.00   
5.32      57.015308  95.000000          412.0   9.954601  ...  19831.25   
5.42      57.015308  57.015308            7.0   8.601350  ...   7750.00   
5.79      57.015308  57.015308            4.0   8.601350  ...   9600.00   
5.93      57.015308  57.015308            9.0  10.251165  ...  15000.00   

                     dti                                                     \
              max  count       mean       std   min      25%    50%     75%   
int_rate                                                                      
5.31      40000.0   81.0  14.645802  8.304183  0.83   8.2300  13.96  18.540   
5.32      40000.0  412.0  15.033617  7.299699  0.23   9.8200  14.01  20.040   
5.42       8000.0    7.0  15.050000  6.734256  5.75   9.5350  18.02  20.125   
5.79      14400.0    4.0  13.947500  5.315872  7.55  11.2475  14.02  16.720   
5.93      25000.0    9.0  17.375556  9.330602  4.39   9.0000  16.42  26.840   

                 
            max  
int_rate         
5.31      48.66  
5.32      45.43  
5.42      22.26  
5.79      20.20  
5.93      28.38  

[5 rows x 56 columns]

##### Để tính khoảng cách Euclidean, chỉ nên sử dụng các numerical feature trong dataset.

Sau đó nhóm dữ liệu theo int_rate và tính giá trị trung bình (mean) của từng feature. Mỗi giá trị int_rate sẽ được biểu diễn bởi một vector đặc trưng đại diện (feature profile).

Các vector này được sử dụng để tìm nhãn gần nhất cho những quan sát chưa có nhãn ở bước tiếp theo.

##### Chuẩn hóa dữ liệu trước khi tính
Bởi vì cột loan_amount có giá trị quá lớn so với các cột còn lại 

In [4]:
from sklearn.preprocessing import StandardScaler

num_cols = labeled_df.select_dtypes(
    include="number"
).columns.drop("int_rate")

scaler = StandardScaler()

labeled_scaled = labeled_df.copy()
unlabeled_scaled = unlabeled_df.copy()

labeled_scaled[num_cols] = scaler.fit_transform(
    labeled_scaled[num_cols]
)

unlabeled_scaled[num_cols] = scaler.transform(
    unlabeled_scaled[num_cols]
)

In [5]:
label_profiles = (
    labeled_scaled
    .groupby("int_rate")[num_cols]
    .mean()
)

##### 4. Find Closest Labeled Observation Using Euclidean Distance

Sau khi xây dựng bảng `label_profiles`, mỗi giá trị `int_rate` được biểu diễn bởi một vector đặc trưng đại diện, bao gồm giá trị trung bình của các numerical feature.

Đối với mỗi quan sát chưa có nhãn, ta trích xuất các numerical feature và tính khoảng cách Euclidean tới từng feature profile trong `label_profiles`.

Giá trị `int_rate` có khoảng cách nhỏ nhất sẽ được xem là nhãn phù hợp nhất cho quan sát đó.

##### Chuẩn hóa dữ liệu trước khi tính
Bởi vì cột loan_amount có giá trị quá lớn so với các cột còn lại 

predicted_labels = []

for _, unlabeled_row in unlabeled_scaled.iterrows(): #duyệt từng dòng chưa có nhãn

    min_distance = float("inf")
    best_label = None

    for int_rate, profile in label_profiles.iterrows(): #duyệt từng feature profile

        distance = 0

        for col in num_cols:
            distance += (
                unlabeled_row[col] - profile[col]
            ) ** 2

        distance = distance ** 0.5

        if distance < min_distance:
            min_distance = distance
            best_label = int_rate

    predicted_labels.append(best_label)

In [6]:
from scipy.spatial.distance import cdist

X = unlabeled_scaled[num_cols].to_numpy()      # (80000, 13)
C = label_profiles[num_cols].to_numpy()        # (P, 13)

dist = cdist(X, C)                             # (80000, P), khoảng cách Euclid
predicted_labels = label_profiles.index[dist.argmin(axis=1)].tolist()

iterrows trả về 2 giá trị(index, row) </br>
Tất cả các feature cùng tham gia tính khoảng cách Euclidean. Sau đó chọn int_rate có tổng khoảng cách nhỏ nhất.
```text
all_util
+
bc_open_to_buy
+
annual_inc
+
loan_amnt
+
dti
+
...
↓
Euclidean Distance
↓
Chọn int_rate gần nhất
```
---


##### Đánh giá sai số thông qua các chỉ số Mean Absolute Error (MAE).

In [7]:
original_labels = df.loc[
    unlabeled_df.index,
    "int_rate"
]

from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(
    original_labels,
    predicted_labels
)

print(mae)

7.214515124999999


In [8]:
comparison = pd.DataFrame({
    "Actual": original_labels.values,
    "Predicted": predicted_labels
})

comparison.head()

,Actual,Predicted
0,10.42,21.97
1,15.05,24.37
2,5.32,28.49
3,15.61,6.62
4,5.32,6.08


In [13]:
print(label_profiles.shape)

(465, 7)


Kết quả cho thấy phương pháp Summary Statistics + Euclidean Distance hoạt động nhưng độ chính xác chưa cao (MAE ≈ 7.4).

Một nguyên nhân có thể là biến mục tiêu `int_rate` có tới 465 giá trị khác nhau, khiến mỗi nhóm chứa rất ít quan sát và các thống kê đại diện chưa thực sự ổn định.

##### Pseudo-Labeling

Pseudo-labeling là một kỹ thuật semi-supervised learning, trong đó mô hình được huấn luyện trên tập dữ liệu đã có nhãn.

Sau khi huấn luyện, mô hình sẽ dự đoán nhãn cho các quan sát chưa có nhãn. Các giá trị dự đoán này được gọi là pseudo labels.

Cuối cùng, pseudo labels được gán lại cho tập dữ liệu chưa có nhãn nhằm mở rộng tập dữ liệu huấn luyện.

In [14]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    random_state=42
)

model.fit(
    labeled_df[num_cols],
    labeled_df["int_rate"]
)

pseudo_labels = model.predict(
    unlabeled_df[num_cols]
)

In [15]:
mae = mean_absolute_error(
    original_labels,
    pseudo_labels
)

print(f"MAE: {mae:.4f}")

MAE: 3.3464


In [16]:
comparison = pd.DataFrame({
    "Actual": original_labels.values,
    "Pseudo_Label": pseudo_labels
})

comparison.head()

,Actual,Pseudo_Label
0,10.42,14.6898
1,15.05,13.0149
2,5.32,10.6492
3,15.61,13.9821
4,5.32,9.7830


##### 6. Lọc nhãn giả, gộp vào tập train và huấn luyện lại (Self-training)

Ở bước 5, toàn bộ pseudo label được tạo ra nhưng **chưa được kiểm tra độ tin cậy**. Nếu đưa hết vào tập train thì các nhãn sai sẽ làm nhiễu mô hình. Quy trình tự học bán giám sát hoàn chỉnh gồm 3 bước:

```text
1. Lọc nhãn giả          → chỉ giữ các nhãn giả đáng tin cậy
        ↓
2. Gộp vào tập train     → nhãn giả đã lọc + tập gốc (labeled_df)
        ↓
3. Huấn luyện lại        → mô hình mới dự đoán cho phần chưa có nhãn còn lại
```

**Làm sao biết nhãn giả nào đáng tin?**

Trong thực tế dữ liệu chưa có nhãn thì không có đáp án nên không thể tính MAE để lọc trực tiếp. Thay vào đó ta dùng **độ bất định (uncertainty)** của mô hình: với Random Forest, mỗi cây đưa ra một dự đoán riêng, nếu các cây dự đoán **gần nhau** (độ lệch chuẩn thấp) thì mô hình "tự tin", nhãn giả đó đáng tin hơn.

Vì ở đây `int_rate` được che có chủ đích nên ta vẫn còn nhãn thật (`original_labels`) và dùng nó **chỉ để kiểm chứng** xem tiêu chí lọc có thực sự giúp MAE thấp đi hay không. Nhãn thật không tham gia vào việc lọc hay huấn luyện.

###### 6.1. Lọc nhãn giả đáng tin cậy

Tính độ bất định của từng pseudo label bằng độ lệch chuẩn giữa các dự đoán của từng cây trong Random Forest.

In [17]:
# Dự đoán của từng cây trên tập chưa có nhãn -> shape (n_trees, n_unlabeled)
X_unlabeled = unlabeled_df[num_cols].to_numpy()
tree_preds = np.stack([tree.predict(X_unlabeled) for tree in model.estimators_])

pseudo_df = pd.DataFrame(
    {
        "pseudo_label": pseudo_labels,            # = trung bình các cây
        "uncertainty": tree_preds.std(axis=0),    # độ lệch chuẩn giữa các cây
        "true_label": original_labels.values,     # chỉ dùng để đánh giá, KHÔNG dùng để lọc
    },
    index=unlabeled_df.index,
)
pseudo_df["abs_error"] = (pseudo_df["pseudo_label"] - pseudo_df["true_label"]).abs()

pseudo_df.head()

,pseudo_label,uncertainty,true_label,abs_error
0,14.6898,5.122909,10.42,4.2698
1,13.0149,3.734609,15.05,2.0351
3,10.6492,4.447805,5.32,5.3292
4,13.9821,4.761929,15.61,1.6279
6,9.7830,4.159416,5.32,4.4630


Kiểm tra xem `uncertainty` có phản ánh đúng độ tin cậy hay không: nếu độ bất định thấp thì sai số thật cũng nên thấp (tương quan dương).

In [18]:
corr = pseudo_df["uncertainty"].corr(pseudo_df["abs_error"], method="spearman")
print(f"Spearman(uncertainty, abs_error) = {corr:.3f}\n")

# MAE của phần nhãn giả được giữ lại theo các mức ngưỡng khác nhau
rows = []
for q in [0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
    thr = pseudo_df["uncertainty"].quantile(q)
    kept = pseudo_df[pseudo_df["uncertainty"] <= thr]
    rows.append({
        "giữ % nhãn giả": int(q * 100),
        "số nhãn giữ lại": len(kept),
        "ngưỡng uncertainty": round(thr, 3),
        "MAE phần giữ lại": round(kept["abs_error"].mean(), 4),
    })

pd.DataFrame(rows)

Spearman(uncertainty, abs_error) = 0.125



,giữ % nhãn giả,số nhãn giữ lại,ngưỡng uncertainty,MAE phần giữ lại
0,20,16000,3.502,2.8770
1,40,32000,3.964,2.9939
2,50,40000,4.175,3.0472
3,60,48000,4.396,3.1009
4,80,64000,4.955,3.2088
5,100,80000,9.904,3.3464


Nếu `uncertainty` phản ánh đúng độ tin cậy (Spearman dương) thì giữ càng ít nhãn, MAE phần giữ lại càng thấp (nhãn càng sạch), nhưng số lượng dữ liệu thêm vào train cũng ít đi. Cần cân bằng giữa **chất lượng** và **số lượng**. Ở đây chọn giữ **50%** nhãn giả có độ bất định thấp nhất (có thể chỉnh `CONFIDENCE_QUANTILE`).

In [19]:
CONFIDENCE_QUANTILE = 0.5   # giữ 50% nhãn giả có độ bất định thấp nhất

threshold = pseudo_df["uncertainty"].quantile(CONFIDENCE_QUANTILE)

reliable_df = pseudo_df[pseudo_df["uncertainty"] <= threshold]    # nhãn giả đáng tin cậy
remaining_df = pseudo_df[pseudo_df["uncertainty"] > threshold]    # chưa đủ tin cậy, để dự đoán lại sau

print(f"Ngưỡng uncertainty : {threshold:.4f}")
print(f"Nhãn giả giữ lại   : {len(reliable_df)}  | MAE = {reliable_df['abs_error'].mean():.4f}")
print(f"Nhãn giả bị loại   : {len(remaining_df)}  | MAE = {remaining_df['abs_error'].mean():.4f}")
print(f"Toàn bộ (chưa lọc) : {len(pseudo_df)}  | MAE = {pseudo_df['abs_error'].mean():.4f}")

Ngưỡng uncertainty : 4.1745
Nhãn giả giữ lại   : 40000  | MAE = 3.0472
Nhãn giả bị loại   : 40000  | MAE = 3.6456
Toàn bộ (chưa lọc) : 80000  | MAE = 3.3464


###### 6.2. Gộp nhãn giả đáng tin cậy vào tập train

Tập train mới = tập gốc đã có nhãn (`labeled_df`) + các quan sát chưa có nhãn nhưng đã được gán nhãn giả đáng tin cậy.

In [20]:
reliable_train = unlabeled_df.loc[reliable_df.index, num_cols].copy()
reliable_train["int_rate"] = reliable_df["pseudo_label"]

augmented_df = pd.concat(
    [labeled_df[list(num_cols) + ["int_rate"]], reliable_train],
    ignore_index=False,
)

print("Tập train gốc      :", len(labeled_df))
print("Nhãn giả được thêm :", len(reliable_train))
print("Tập train mới      :", len(augmented_df))

Tập train gốc      : 20000
Nhãn giả được thêm : 40000
Tập train mới      : 60000


###### 6.3. Huấn luyện lại mô hình

Huấn luyện lại Random Forest trên tập train đã được mở rộng, sau đó dự đoán cho phần chưa có nhãn còn lại (`remaining_df`).

Để so sánh công bằng, cả mô hình cũ (chỉ train trên tập gốc) và mô hình mới đều được đánh giá **trên cùng phần `remaining_df`**.

In [21]:
model_v2 = RandomForestRegressor(random_state=42)
model_v2.fit(augmented_df[num_cols], augmented_df["int_rate"])

# Dự đoán lại cho phần chưa có nhãn còn lại
remaining_pred_v2 = model_v2.predict(unlabeled_df.loc[remaining_df.index, num_cols])

mae_remaining_v1 = mean_absolute_error(remaining_df["true_label"], remaining_df["pseudo_label"])
mae_remaining_v2 = mean_absolute_error(remaining_df["true_label"], remaining_pred_v2)

print(f"MAE trên phần còn lại - mô hình cũ (chỉ tập gốc)     : {mae_remaining_v1:.4f}")
print(f"MAE trên phần còn lại - mô hình mới (sau self-train) : {mae_remaining_v2:.4f}")

# Gộp lại thành nhãn cuối cùng cho toàn bộ 200 quan sát chưa có nhãn
final_labels = pd.concat([
    reliable_df["pseudo_label"],
    pd.Series(remaining_pred_v2, index=remaining_df.index),
]).loc[unlabeled_df.index]

mae_self_training = mean_absolute_error(original_labels, final_labels)
print(f"\nMAE toàn bộ tập chưa có nhãn - pseudo-label 1 lần  : {mean_absolute_error(original_labels, pseudo_labels):.4f}")
print(f"MAE toàn bộ tập chưa có nhãn - self-training 1 vòng : {mae_self_training:.4f}")

MAE trên phần còn lại - mô hình cũ (chỉ tập gốc)     : 3.6456
MAE trên phần còn lại - mô hình mới (sau self-train) : 3.6577

MAE toàn bộ tập chưa có nhãn - pseudo-label 1 lần  : 3.3464
MAE toàn bộ tập chưa có nhãn - self-training 1 vòng : 3.3525


###### 6.4. Lặp lại nhiều vòng (Iterative Self-training)

Mỗi vòng ta lọc nhãn giả tin cậy → gộp vào train → huấn luyện lại, rồi tiếp tục với phần còn chưa có nhãn. Vòng cuối cùng, mô hình dự đoán nốt phần còn lại.

In [22]:
def self_training(labeled, unlabeled, features, n_rounds=3, quantile=0.5, random_state=42):
    """Self-training cho bài toán hồi quy.
    Mỗi vòng: train -> dự đoán -> lọc nhãn giả có uncertainty thấp -> gộp vào train.
    Trả về (nhãn giả cho toàn bộ unlabeled, lịch sử các vòng)."""
    X_train = labeled[features].copy()
    y_train = labeled["int_rate"].copy()
    pool = unlabeled[features].copy()
    pseudo = pd.Series(index=unlabeled.index, dtype=float)
    history = []

    for r in range(1, n_rounds + 1):
        if pool.empty:
            break

        rf = RandomForestRegressor(random_state=random_state, n_jobs=-1)
        rf.fit(X_train, y_train)

        preds = np.stack([t.predict(pool.to_numpy()) for t in rf.estimators_])
        mean_pred, unc = preds.mean(axis=0), preds.std(axis=0)

        keep = unc <= np.quantile(unc, quantile)          # 1. lọc nhãn giả tin cậy
        kept_idx = pool.index[keep]
        pseudo.loc[kept_idx] = mean_pred[keep]

        X_train = pd.concat([X_train, pool.loc[kept_idx]])  # 2. gộp vào tập train
        y_train = pd.concat([y_train, pd.Series(mean_pred[keep], index=kept_idx)])
        pool = pool.drop(index=kept_idx)                   # phần còn lại chờ vòng sau

        history.append({"vòng": r, "nhãn giả thêm vào": len(kept_idx),
                        "tập train": len(X_train), "còn chưa gán": len(pool)})

    # 3. huấn luyện lại lần cuối, dự đoán nốt phần còn lại
    if not pool.empty:
        rf = RandomForestRegressor(random_state=random_state, n_jobs=-1)
        rf.fit(X_train, y_train)
        pseudo.loc[pool.index] = rf.predict(pool)

    return pseudo, pd.DataFrame(history)


pseudo_iter, history = self_training(
    labeled_df, unlabeled_df, num_cols, n_rounds=3, quantile=0.5
)

mae_iterative = mean_absolute_error(original_labels, pseudo_iter.loc[unlabeled_df.index])
print(f"MAE (self-training lặp 3 vòng): {mae_iterative:.4f}")
history

MAE (self-training lặp 3 vòng): 3.3481


,vòng,nhãn giả thêm vào,tập train,còn chưa gán
0,1,40000,60000,40000
1,2,20000,80000,20000
2,3,10000,90000,10000


### Comparison of Labeling Methods


Kết quả cho thấy phương pháp Pseudo-Labeling + Self Training đạt sai số thấp hơn đáng kể so với phương pháp sử dụng Summary Statistics kết hợp Euclidean Distance.

Nguyên nhân là các mô hình học máy có khả năng học được các mối quan hệ phi tuyến giữa các feature và biến mục tiêu `int_rate`, trong khi phương pháp Euclidean Distance chỉ dựa trên độ tương đồng giữa các thống kê mô tả.

Do đó, Pseudo-Labeling + Self Training được đánh giá là phương pháp hiệu quả hơn trên dataset này.

###### Tổng hợp kết quả 

In [23]:
mae_euclid = mean_absolute_error(original_labels, predicted_labels)
mae_pseudo = mean_absolute_error(original_labels, pseudo_labels)

results = pd.DataFrame({
    "Method": [
        "Summary Statistics + Euclidean Distance",
        "Pseudo-Labeling (Random Forest, không lọc)",
        "Self-training 1 vòng (lọc + gộp + train lại)",
        "Self-training lặp 3 vòng",
    ],
    "MAE": [mae_euclid, mae_pseudo, mae_self_training, mae_iterative],
}).round(4)

results

,Method,MAE
0,Summary Statistics + Euclidean Distance,7.2145
1,"Pseudo-Labeling (Random Forest, không lọc)",3.3464
2,Self-training 1 vòng (lọc + gộp + train lại),3.3525
3,Self-training lặp 3 vòng,3.3481


**Nhận xét**

- Bước **lọc nhãn giả**: so sánh MAE của phần được giữ lại với phần bị loại ở mục 6.1. Nếu phần giữ lại có MAE thấp hơn thì tiêu chí lọc theo độ bất định là hợp lý; nếu hai phần gần như bằng nhau thì nên thử tiêu chí khác (ví dụ độ bất định từ nhiều mô hình khác nhau).
- Sau khi gộp nhãn giả tin cậy vào tập train và huấn luyện lại, mô hình được đánh giá trên cùng phần dữ liệu còn lại để so sánh công bằng với mô hình ban đầu.
- Lưu ý: self-training **không đảm bảo** luôn cải thiện MAE. Nhãn giả do chính mô hình tạo ra nên lượng thông tin mới thêm vào có hạn, và nếu nhãn giả sai vẫn lọt qua bộ lọc thì sai số có thể bị khuếch đại. Với tập nhỏ (800 labeled) kết quả có thể chỉ chênh lệch ít, nên cần thử nhiều giá trị `CONFIDENCE_QUANTILE` và số vòng lặp.